## Quiet the logs

`scarlets/utils/RedisLogger.py` calls `logging.basicConfig(level=DEBUG)` at
**import time**, which sets the *root* logger to DEBUG for the whole kernel -
so every library that logs (redis-py, httpx, asyncio) starts emitting DEBUG
into your cell output as soon as anything imports `scarlets`.

Run this first. It configures the root logger before `scarlets` gets the
chance, which makes that `basicConfig` call a no-op (`basicConfig` does
nothing when the root logger already has handlers).

If you ever see lines like `Failed to enable maintenance notifications:
unknown subcommand 'MAINT_NOTIFICATIONS'` - those are harmless. redis-py 8.x
probes for a Redis 8 maintenance feature on every new connection, Redis 7.x
replies "unknown subcommand", and the client logs it at DEBUG and carries on
with a perfectly normal connection.

In [1]:
import logging

# Claim the root logger first. logging.basicConfig() does nothing if the
# root logger already has handlers, so configuring it here makes
# RedisLogger's own basicConfig(level=DEBUG) - which runs when scarlets is
# imported further down - a no-op instead of a takeover.
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

# Belt and braces: if something already imported scarlets in this kernel
# (a re-run, or an earlier notebook), the call above was itself the no-op,
# so set the level explicitly too.
logging.getLogger().setLevel(logging.WARNING)
for _noisy in ("redis", "httpx", "httpx2", "httpcore", "httpcore2",
               "urllib3", "asyncio", "mcp"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

print("log level: WARNING (set INFO or DEBUG here if you want the detail back)")

log level: WARNING (set INFO or DEBUG here if you want the detail back)


# 06 — Asking a real agent fleet a plain-English question

Every notebook so far ran everything inside this one kernel process. This
one is different: it works against **real, separate `scarlet-agents`
containers** (one head, one or more workers), waits for them to come
online, then asks a question in plain English - the head's LLM decides
which `scarlet_agentic_harness` skills to call (here: `sum`, twice, with
different transforms) and reports back a natural-language answer, having
actually dispatched to and coordinated with the worker containers over
Redis.

Each worker's private data is its own CSV file, written by this notebook
into a directory it shares with the workers - real separate containers, so
each just gets its own `CSV_PATH`; no in-process trickery needed to keep
them from stepping on each other's data.

## Two ways the fleet gets deployed

This notebook does not care which one you use:

- **Locally (default)** - `examples/notebooks/docker-compose.yml` declares
  the agent containers under an `agents` profile, which a plain
  `docker compose up` skips. This notebook brings them in on demand.
- **Through Gustavo** - deploy the app configs in `examples/notebooks/
  gustavo-app.yaml` instead. Gustavo owns the fleet's lifecycle and sets
  `GUSTAVO_MANAGED_AGENTS=true`, so this notebook waits for the agents
  rather than launching (or tearing down) anything itself.

Either way the notebook talks to the fleet the same way: over Redis for
liveness, over the head's MCP endpoint to ask the question.

## Parameters

Everything that names an agent lives here, all overridable by environment
variable - which is what lets the same notebook run unmodified in both
deployments.

An agent's identity on the bus is `f"{APP_ID}_{NODE_ADDRESS}"`
(`HarnessConfig.agent_id`). The head and the workers are **separate
deployments with separate `APP_ID`s**, so there is no single `APP_ID` to
compose every id from - hence `HEAD_APP_ID` and `WORKER_APP_ID` as
distinct settings. Under Gustavo each app's `APP_ID` is forced to its own
app name (`gustavo/api/routers/apps.py`), and the compose file sets the
same two values explicitly, so the two deployments agree.

`EXPECTED_AGENT_IDS` overrides the derivation entirely with a verbatim
comma-separated list, for a deployment whose ids don't follow from these
settings at all.

In [2]:
import os, time, json, subprocess, random

# ─── identity ────────────────────────────────────────────────────────────
# Head and workers are separate deployments with their own APP_IDs, so
# each id needs its own prefix - one shared APP_ID cannot produce both.
HEAD_APP_ID   = os.environ.get("HEAD_APP_ID", "scarlet-agent-head")
WORKER_APP_ID = os.environ.get("WORKER_APP_ID", "scarlet-agent-worker")

HEAD_NODE    = os.environ.get("HEAD_NODE_ADDRESS", "head")
WORKER_NODES = [n.strip() for n in
                os.environ.get("WORKER_NODE_ADDRESSES", "worker1").split(",")
                if n.strip()]

# The bus the harness head registers on, and where its MCP server is
# reachable. Under compose the host is the service name; under Gustavo it
# is the container name Gustavo creates (e.g. scarlet-agent-head-1).
HEAD_BUS      = os.environ.get("HEAD_BUS", f"{HEAD_APP_ID}_headagent")
HEAD_HOST     = os.environ.get("HEAD_HOST", "agent-head")
HEAD_MCP_PORT = os.environ.get("HEAD_MCP_PORT", "8090")

# ─── who we expect on the bus ────────────────────────────────────────────
_explicit_ids = os.environ.get("EXPECTED_AGENT_IDS", "").strip()
if _explicit_ids:
    EXPECTED_AGENT_IDS = {i.strip() for i in _explicit_ids.split(",") if i.strip()}
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = EXPECTED_AGENT_IDS - {HEAD_AGENT_ID}
else:
    HEAD_AGENT_ID      = f"{HEAD_APP_ID}_{HEAD_NODE}"
    WORKER_AGENT_IDS   = {f"{WORKER_APP_ID}_{n}" for n in WORKER_NODES}
    EXPECTED_AGENT_IDS = {HEAD_AGENT_ID} | WORKER_AGENT_IDS

# ─── deployment mode ─────────────────────────────────────────────────────
# false (the default) means this notebook owns the fleet: it brings the
# compose "agents" profile up below and tears it down at the end. true
# means Gustavo deployed the agents and owns them - this notebook then
# only waits and observes, and the teardown cell is a no-op.
GUSTAVO_MANAGED_AGENTS = os.environ.get("GUSTAVO_MANAGED_AGENTS", "false").lower() == "true"
COMPOSE_FILE = os.environ.get("COMPOSE_FILE_PATH", "/workspace/docker-compose.yml")

print(f"head app id   : {HEAD_APP_ID}")
print(f"worker app id : {WORKER_APP_ID}")
print(f"expecting     : {sorted(EXPECTED_AGENT_IDS)}")
print(f"head bus      : {HEAD_BUS!r}")
print(f"fleet managed : {'Gustavo' if GUSTAVO_MANAGED_AGENTS else 'this notebook (docker compose)'}")

head app id   : loop-agent-head
worker app id : loop-agent-worker
expecting     : ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']
head bus      : 'loop_headagent'
fleet managed : Gustavo


## Generate each worker's private data

One CSV per worker, written into a directory this container shares with
them. The workers read them through `data_connectors.CsvConnector`, which
expects a single `value` column - see
`harness/scarlet_agentic_harness/skills/local_data.py`.

**This runs in both deployments**, because in both the workers read from a
path this notebook can also write to: a shared Docker volume under compose,
and a shared host directory (`/tmp/data` mounted into both the notebook and
each worker) under Gustavo.

That matters more than it sounds. If `CSV_PATH` is set on a worker but the
file is missing, `pd.read_csv` raises inside `contribute()`, the readiness
signal is never sent, and the coordinator waits out its full timeout with
no error anywhere the Composer can show you - the dispatch simply never
completes. Writing the data here is what keeps that from happening.

The seed is fixed, so the variance the agent reports at the end is
reproducible run to run.

In [3]:
DATA_DIR = os.environ.get("DATA_DIR", "/data")

def write_worker_csv(path, values):
    with open(path, "w") as f:
        f.write("value\n")
        for v in values:
            f.write(f"{v}\n")

rng = random.Random(20260928)   # fixed seed: reproducible variance
os.makedirs(DATA_DIR, exist_ok=True)
for i, node in enumerate(WORKER_NODES, start=1):
    # Each worker gets a different distribution, so a fleet-wide variance
    # is genuinely a property of the combined data rather than of any one
    # worker's.
    values = [round(rng.gauss(20 + 2 * i, 1.5), 2) for _ in range(12)]
    path = f"{DATA_DIR}/{node}.csv"
    write_worker_csv(path, values)
    print(f"{node}: wrote {len(values)} readings to {path}")

_DESCRIPTION = "Sensor readings held by this worker. One numeric column: value."

# --- local config for each worker -------------------------------------
# This sprint retired CSV_PATH. A worker no longer gets a path from its
# environment: it reads its own ~/.scarlet/config.yaml, profiles whatever
# sources it finds there at boot, and writes its own SQL against them.
#
# HOME for each worker points into this shared volume (see the compose
# file), so the path below is the worker's *real* ~/.scarlet/config.yaml -
# there is no SCARLET_LOCAL_CONFIG override in play.
import yaml as _yaml

for _node in WORKER_NODES:
    _csv = f"{DATA_DIR}/{_node}.csv"
    _scarlet_dir = os.path.join(DATA_DIR, _node, ".scarlet")
    os.makedirs(_scarlet_dir, exist_ok=True)
    with open(os.path.join(_scarlet_dir, "config.yaml"), "w") as _fh:
        _yaml.safe_dump({"sources": [{
            "name": f"{_node}_readings",
            "type": "csv",
            "mode": "local",
            "description": _DESCRIPTION,
            "path": _csv,
        }]}, _fh, sort_keys=False)
print(f"wrote {len(WORKER_NODES)} worker configs under {DATA_DIR}/<worker>/.scarlet/")


worker1: wrote 12 readings to /data/worker1.csv
worker2: wrote 12 readings to /data/worker2.csv
worker3: wrote 12 readings to /data/worker3.csv
worker4: wrote 12 readings to /data/worker4.csv
wrote 4 worker configs under /data/<worker>/.scarlet/


## Bring the fleet up

One `docker compose --profile agents up -d` against the same compose file
that started this notebook - bind-mounted in at `/workspace`, driving the
host's Docker daemon through the mounted socket. The compose project name
is pinned in that file, so these containers join the network the
already-running Redis is on rather than starting a second, isolated stack.

`up -d` is idempotent: re-running this cell leaves anything already running
alone and starts only what is missing, which is also what makes a
partially-dead fleet self-healing.

When `GUSTAVO_MANAGED_AGENTS=true` this cell launches nothing at all.

In [4]:
# Compose SERVICE names, derived from the same WORKER_NODES the identity
# above uses - so the services that get started and the agents that get
# waited for can never disagree about how many workers there are.
AGENT_SERVICES = ["agent-head"] + [f"agent-{n}" for n in WORKER_NODES]


def compose(*args):
    """Run a compose command scoped to the agent services only.

    Failures print what Compose actually said: check=True on its own would
    raise CalledProcessError with the diagnostic buried in .stderr, which
    Jupyter renders as a bare traceback giving no hint what was wrong.
    """
    result = subprocess.run(
        ["docker", "compose", "-f", COMPOSE_FILE, "--profile", "agents", *args],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            f"`docker compose {' '.join(args)}` failed with exit "
            f"{result.returncode} - see the Compose output above."
        )
    return result


if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - not launching anything.")
    print("If the wait below times out, check the Gustavo deployment.")
else:
    # Fail fast on a missing LLM endpoint. The head runs mcp_server.py,
    # which exits immediately without LLM_BASE_URL - unlike __main__.py's
    # REPL it has no manual-dispatch fallback. Without this check the
    # container restart-loops, the workers come up fine, and the only
    # symptom is the liveness wait below timing out after two minutes
    # naming the head - pointing at compose logs rather than at the actual
    # cause, which is an unset value in .env.
    missing_llm = [k for k in ("LLM_BASE_URL", "LLM_MODEL")
                   if not os.environ.get(k, "").strip()]
    if missing_llm:
        raise RuntimeError(
            f"{', '.join(missing_llm)} not set - the agent head cannot start "
            f"without them.\nSet these in examples/notebooks/.env and restart "
            f"the stack (docker compose up -d), then re-run this notebook."
        )

    print("bringing up:", ", ".join(AGENT_SERVICES))
    # --no-deps: bring up ONLY these services, never their depends_on.
    # Redis is already running (this notebook could not be talking to you
    # otherwise), and without this Compose may decide the running Redis is
    # out of date and recreate it - dropping the bus out from under any
    # agent already registered on it. The bring-up should only ever add
    # containers, never disturb the stack.
    #compose("up", "-d", "--no-deps", *AGENT_SERVICES)
    print("containers started - waiting for them to register below")

Gustavo owns this fleet - not launching anything.
If the wait below times out, check the Gustavo deployment.


## Wait for the fleet to come online

`up -d` returns as soon as the containers exist, well before the harness
inside them has connected to Redis and registered - and under Gustavo this
notebook has no idea when the rollout will land. So this is a real wait,
not a fixed sleep: poll `GatherStatus()` on the head bus until every
expected agent is there.

**Liveness is judged on freshness, not presence.** A registration record
carries a `ts` refreshed by the agent's heartbeat every 30 s, and an agent
counts as live only if its `ts` is recent. Historically this was essential:
records were written with no TTL, so a dead agent's entry sat in Redis
forever and a presence-only check would happily match a previous run's
corpses. Registrations now expire on their own, but the freshness filter
stays as belt-and-braces - it is also what protects you against agents
still running an older image.

In [5]:
from scarlets.messaging import Messenger

# 3x the 30s heartbeat interval: tolerates one missed beat without a false
# negative, while a fleet dead longer than this is correctly seen as gone.
FRESH_SECONDS = 90

_probe = Messenger(HEAD_BUS, agentId="notebook_probe")

def live_agents():
    """Agent ids on the head bus whose heartbeat is recent enough to trust."""
    now = time.time()
    return {
        agent_id
        for agent_id, record in _probe.GatherStatus().items()
        if now - record.get("ts", 0) < FRESH_SECONDS
    }

def wait_until_live(timeout):
    deadline = time.time() + timeout
    while True:
        online  = live_agents()
        missing = EXPECTED_AGENT_IDS - online
        if not missing:
            print(f"fleet online: {sorted(EXPECTED_AGENT_IDS)}")
            return online
        if time.time() >= deadline:
            where = ("the Gustavo deployment" if GUSTAVO_MANAGED_AGENTS
                     else f"docker compose -f {COMPOSE_FILE} --profile agents logs")
            raise TimeoutError(
                f"after {timeout}s these agents are still not live on {HEAD_BUS!r}: "
                f"{sorted(missing)}\nSeen on the bus: {sorted(live_agents())}\n"
                f"Check {where}."
            )
        time.sleep(2)

# A remote Gustavo rollout is slower than a local container start.
wait_until_live(timeout=180 if GUSTAVO_MANAGED_AGENTS else 120)

fleet online: ['loop-agent-head_head', 'loop-agent-worker_worker1', 'loop-agent-worker_worker2', 'loop-agent-worker_worker3', 'loop-agent-worker_worker4']


{'__composer_api__',
 'loop-agent-head_head',
 'loop-agent-worker_worker1',
 'loop-agent-worker_worker2',
 'loop-agent-worker_worker3',
 'loop-agent-worker_worker4',
 'notebook_probe'}

## Ask a plain-English question

The head decides, on its own, to call `sum` twice (`transform=identity` for
Sigma(x), `transform=square` for Sigma(x^2)) across the workers' real CSV
data, then composes mean/variance/stddev itself before replying - no
distributed "variance" skill exists, by design (see `sum.py`'s own
docstring).

This notebook's own process never calls the LLM. It only speaks MCP to the
head, which holds the LLM credentials; the workers never see them.

In [6]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_HOST}:{HEAD_MCP_PORT}/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text


# Awaited directly rather than wrapped in asyncio.run(). The Jupyter kernel
# is already running an event loop, and asyncio.run() refuses to start a
# second one inside an existing one ("asyncio.run() cannot be called from a
# running event loop"). IPython supports top-level await for exactly this,
# so the coroutine joins the loop already running instead of owning one.
#
# This makes the cell notebook-only: `await` at top level is a syntax error
# in a plain script, where asyncio.run() would be the correct form instead.
answer = await ask_scarlet_agent(
    "What are the sum, mean, and variance of the readings held across all "
    "currently online workers?"
)
print(answer)

The sum, mean, and variance of the readings held across all currently online workers are:

- Sum: 1181.23
- Mean: 24.609 (rounded to 3 decimal places)
- Variance: 8.491 (rounded to 3 decimal places)


## Teardown

Stops and removes the agent containers, leaving Redis, the Composer UI, and
this notebook running - so you can re-run from the top without restarting
the whole stack.

`docker compose down` from `examples/notebooks/` on the host takes the rest
down when you're finished.

Under Gustavo this is deliberately a no-op: those containers belong to that
deployment, and a tutorial notebook has no business removing them.

In [7]:
if GUSTAVO_MANAGED_AGENTS:
    print("Gustavo owns this fleet - leaving it alone.")
else:
    # `rm -s -f <services>`, never a bare `compose down`: down would take
    # the whole project with it - Redis, the Composer UI, and the jupyter
    # container this notebook is running inside. Stop and remove only the
    # agent containers, leaving the stack you started with intact.
    #compose("rm", "-s", "-f", *AGENT_SERVICES)
    print("agent containers stopped and removed")

Gustavo owns this fleet - leaving it alone.
